## Needs Revisting Calling tools continuously

In [25]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)
import os

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
base_url = "https://openrouter.ai/api/v1"
openai = OpenAI(api_key=openrouter_api_key, base_url=base_url)


In [7]:
todos = []
completed = []

In [9]:
def get_todo_report() -> str:
    """
    Generates and displays a formatted report of the current todos.

    Returns:
        str: The formatted todo list, with completed items struck through in green.
    """
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [ ]:
get_todo_report()

In [ ]:
def create_todos(descriptions: list[str]) -> str:
    """
    Adds a list of todo item descriptions to the global todo list.
    Marks all newly added todos as not completed.
    Returns the updated formatted todo report.

    Args:
        descriptions (list[str]): A list of descriptions for new todo items.

    Returns:
        str: The formatted todo list after adding the new items.
    """
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [12]:
def mark_complete(index: int, completion_notes: str) -> str:
    """
    Marks the todo item at the given index as completed and displays completion notes.

    Args:
        index (int): The 1-based index of the todo item to mark as completed.
        completion_notes (str): Notes or comments regarding the completion of the task.

    Returns:
        str: The formatted todo report after marking the item as completed,
             or an error message if the index is invalid.
    """
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [ ]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

In [ ]:
mark_complete(1, "bought")

In [15]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [16]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [17]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [22]:
def handle_tool_calls(tool_calls):
    """
    Handles tool calls from the language model by executing corresponding Python functions
    and packaging their results for insertion in the conversation history.

    Args:
        tool_calls (list): A list of tool call objects, each representing a function call
                           as requested by the language model.

    Returns:
        list: A list of message dictionaries with role 'tool', content as the JSON-encoded 
              result from each function call, and the tool_call_id for threading.
    """
    results = []
    for tool_call in tool_calls:
        # Retrieve the function name to be called as specified by the LLM
        tool_name = tool_call.function.name

        # Parse the arguments for the function call (as a dict)
        arguments = json.loads(tool_call.function.arguments)

        # Get the function object from the global namespace
        tool = globals().get(tool_name)

        # Call the tool with the parsed arguments, or return empty dict if not found
        result = tool(**arguments) if tool else {}

        # Package the result as a message dict for conversation continuation
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

In [23]:
def loop(messages):
    """
    Repeatedly sends messages to the language model until a final (non-tool-calls) response is received.

    This function handles tool call requests by executing corresponding tools/functions and updating
    the message history accordingly. When the language model responds with a final answer (not requiring
    further tool calls), it displays the content.

    Args:
        messages (list): A conversation history of message dictionaries, used as input to the LLM.

    Returns:
        None. The function displays (via the show function) the final message content from the LLM.
    """
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-5.2",
            messages=messages,
            tools=tools,
            reasoning_effort="none"
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)